# 09 — CatBoost Training
This notebook trains a **CatBoost** model. CatBoost is particularly good at handling categorical features and preventing overfitting using symmetric trees.

**Outputs:**
1. `submissions/catboost_submission.csv` — Ready to submit to Kaggle.
2. `pickles/catboost_oof.pkl` — Out-Of-Fold probabilities for future ensembling.
3. `pickles/catboost_test_preds.pkl` — Test probabilities for ensembling.

In [2]:
# Install CatBoost if needed
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set your project path on Drive
DRIVE_PATH = '/content/drive/MyDrive/santander-customer-satisfaction/'
PICKLE_DIR = DRIVE_PATH + 'pickles/'
SUBMIT_DIR = DRIVE_PATH + 'submissions/'

import os
import pandas as pd
import numpy as np
import pickle
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

print('Setup complete.')

Mounted at /content/drive
Setup complete.


## 1. Load Data & Shared CV Folds

In [4]:
# Load clean datasets
# train = pd.read_pickle(f'{PICKLE_DIR}train_clean.pkl')
# test  = pd.read_pickle(f'{PICKLE_DIR}test_clean.pkl')
train = pd.read_pickle(f'{PICKLE_DIR}train_advanced.pkl')
test = pd.read_pickle(f'{PICKLE_DIR}test_advanced.pkl')

# Separate features and target
X = train.drop(columns=['TARGET', 'ID'], errors='ignore')
y = train['TARGET']
test_features = test.drop(columns=['ID'], errors='ignore')

# Ensure column order matches
test_features = test_features[X.columns]

# Load identical CV folds
with open(f'{PICKLE_DIR}cv_fold_indices.pkl', 'rb') as f:
    cv_folds = pickle.load(f)

print(f"Loaded {len(cv_folds)} CV folds.")

Loaded 5 CV folds.


## 2. Train CatBoost (GPU Optimized)

In [6]:
import numpy as np
import pandas as pd
import pickle
from catboost import CatBoostClassifier, Pool # Ensure CatBoost is imported here too, as it's used in this cell
from sklearn.metrics import roc_auc_score # Ensure roc_auc_score is imported here too

# --- Ensuring data and setup variables are defined for this cell's execution ---
# Copied from mount-drive and load-logic for robustness
DRIVE_PATH = '/content/drive/MyDrive/santander-customer-satisfaction/'
PICKLE_DIR = DRIVE_PATH + 'pickles/'

# Load clean datasets
train = pd.read_pickle(f'{PICKLE_DIR}train_clean.pkl')
test  = pd.read_pickle(f'{PICKLE_DIR}test_clean.pkl')

# Separate features and target
X = train.drop(columns=['TARGET', 'ID'], errors='ignore')
y = train['TARGET']
test_features = test.drop(columns=['ID'], errors='ignore')

# Ensure column order matches
test_features = test_features[X.columns]

# Load identical CV folds
with open(f'{PICKLE_DIR}cv_fold_indices.pkl', 'rb') as f:
    cv_folds = pickle.load(f)
# --- End of added setup code ---

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_aucs = []

# cat_params = {
#     'iterations': 2000,
#     'learning_rate': 0.05,
#     'depth': 6,
#     'eval_metric': 'AUC',
#     'random_seed': 42,
#     'task_type': 'GPU', # Use GPU for speed
#     'verbose': 200,
#     'early_stopping_rounds': 100
# }
# OPTIMIZED CATBOOST PARAMETERS
cat_params = {
    'iterations': 2000,
    'learning_rate': 0.05113275824082385,
    'depth': 6,
    'l2_leaf_reg': 7.658112640634797,
    'random_strength': 8.934300157134556e-06,
    'bagging_temperature': 5.277232910748792,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'task_type': 'CPU', # Changed from GPU to CPU to resolve CUDA driver issue
    'verbose': 200,
    'early_stopping_rounds': 100
}


print("Starting CatBoost Training...")
print("-" * 40)

for fold_dict in cv_folds:
    fold = fold_dict['fold']
    train_idx = fold_dict['train_idx']
    val_idx = fold_dict['val_idx']

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    # OOF Prediction
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds

    fold_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(fold_auc)
    print(f"Fold {fold} AUC: {fold_auc:.5f}")

    # Test Prediction
    test_preds += model.predict_proba(test_features)[:, 1] / len(cv_folds)

print("-" * 40)
print(f"Mean CV AUC: {np.mean(fold_aucs):.5f} \u00b1 {np.std(fold_aucs):.5f}")

Starting CatBoost Training...
----------------------------------------
0:	test: 0.7918767	best: 0.7918767 (0)	total: 157ms	remaining: 5m 13s
200:	test: 0.8281397	best: 0.8281405 (199)	total: 7.37s	remaining: 1m 5s
400:	test: 0.8304612	best: 0.8304612 (400)	total: 16.2s	remaining: 1m 4s
600:	test: 0.8308503	best: 0.8310166 (544)	total: 22.6s	remaining: 52.7s
800:	test: 0.8313756	best: 0.8316261 (738)	total: 31.5s	remaining: 47.2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8316260659
bestIteration = 738

Shrink model to first 739 iterations.
Fold 0 AUC: 0.83163
0:	test: 0.8147861	best: 0.8147861 (0)	total: 41ms	remaining: 1m 21s
200:	test: 0.8413077	best: 0.8413079 (199)	total: 8.13s	remaining: 1m 12s
400:	test: 0.8420279	best: 0.8422571 (335)	total: 15.4s	remaining: 1m 1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8422570727
bestIteration = 335

Shrink model to first 336 iterations.
Fold 1 AUC: 0.84226
0:	test: 0.8098356	best: 0.809835

## 3. Save OOF & Test Predictions

In [7]:
# Save OOF
oof_df = pd.DataFrame({'ID': train['ID'], 'cat_pred': oof_preds})
oof_df.to_pickle(f'{PICKLE_DIR}catboost_oof.pkl')

# Save Test Preds
test_preds_df = pd.DataFrame({'ID': test['ID'], 'cat_pred': test_preds})
test_preds_df.to_pickle(f'{PICKLE_DIR}catboost_test_preds.pkl')

# Save Submission
sub = pd.DataFrame({'ID': test['ID'], 'TARGET': test_preds})
sub.to_csv(f'{SUBMIT_DIR}catboost_submission.csv', index=False)

print("✅ CatBoost predictions saved successfully!")

✅ CatBoost predictions saved successfully!
